# Step 1: people and sky head on SAM 2.1 tiny `image_embed` (Colab)

README section 5, step 1. Scripts: `scripts/poc/people_sky_head/` (see its README for the
objective and the measurements).

Runtime: **GPU** (T4 or better). The run downloads COCO 2017 panoptic annotations and only the
photos with an allowed Flickr license, caches `image_embed` (2 MB per photo) on the local disk of
the runtime, trains the head, and writes the checkpoints and `people_sky_head.onnx` to Google
Drive. If the runtime is reset, run all cells again: the download and the encoder continue
where they stopped while the runtime disk exists; training continues with `RESUME = True`.

With `TRAIN = False`, the notebook only prepares and encodes the validation split and
measures the existing `run/best.pt` on Drive with `evaluate_head.py` (about 15 minutes).

In [ ]:
# Settings.
REPO_URL = "https://github.com/zidage/alcedo-ai-mask-lab.git"
REPO_BRANCH = "main"
SAM2_COMMIT = "2b90b9f5ceec907a1c18123530e92e794ad901a4"
CHECKPOINT_URL = "https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_tiny.pt"

# COCO license ids: 1 BY-NC-SA, 2 BY-NC, 3 BY-NC-ND, 4 BY, 5 BY-SA, 6 BY-ND,
# 7 no known copyright restrictions, 8 United States Government Work.
# Training uses only licenses that permit commercial use (README decision U2).
TRAIN_LICENSES = "4,5,7,8"
TRAIN_MAX_IMAGES = 0      # 0 = all images with an allowed license
VAL_LICENSES = "all"      # validation photos are only measured, not trained on
VAL_MAX_IMAGES = 2500

EPOCHS = 20
BATCH_SIZE = 32
RESUME = False
# False: no train split and no training; only measure the existing run/best.pt on Drive
# (evaluate_head.py) with the measurements of the current head_metrics.py.
TRAIN = True

WORK = "/content/work"
DRIVE_OUT = "/content/drive/MyDrive/alcedo_people_sky_head"

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h /content
!free -g

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
os.makedirs(DRIVE_OUT, exist_ok=True)

In [ ]:
import os
os.makedirs(WORK, exist_ok=True)
%cd {WORK}
if not os.path.isdir("alcedo-ai-mask-lab"):
    !git clone --branch {REPO_BRANCH} {REPO_URL} alcedo-ai-mask-lab
else:
    !git -C alcedo-ai-mask-lab fetch origin {REPO_BRANCH} && git -C alcedo-ai-mask-lab checkout -B {REPO_BRANCH} origin/{REPO_BRANCH}
if not os.path.isdir("sam2"):
    !git clone https://github.com/facebookresearch/sam2.git sam2
!git -C sam2 checkout -q {SAM2_COMMIT} && git -C sam2 rev-parse HEAD
!git -C alcedo-ai-mask-lab log -1 --oneline
!pip install -q -r alcedo-ai-mask-lab/scripts/poc/people_sky_head/requirements_colab.txt
!wget -q -nc {CHECKPOINT_URL}; sha256sum sam2.1_hiera_tiny.pt

## COCO 2017 panoptic annotations

`panoptic_annotations_trainval2017.zip` (person and `sky-other-merged` segments) and the caption
files of `annotations_trainval2017.zip` (the `images` list with the Flickr license of each photo).
COCO annotations are CC BY 4.0.

In [ ]:
%cd {WORK}
!mkdir -p coco/annotations
!wget -q -nc http://images.cocodataset.org/annotations/panoptic_annotations_trainval2017.zip
!wget -q -nc http://images.cocodataset.org/annotations/annotations_trainval2017.zip
!unzip -q -n -j panoptic_annotations_trainval2017.zip -d coco/annotations
!unzip -q -n -j annotations_trainval2017.zip annotations/captions_train2017.json annotations/captions_val2017.json -d coco/annotations
!cd coco && unzip -q -n annotations/panoptic_train2017.zip && unzip -q -n annotations/panoptic_val2017.zip
!ls coco coco/annotations && ls coco/panoptic_train2017 | wc -l

## Targets and photos

Selects the photos by license, downloads only these photos, and writes the per-cell targets.
The first lines of the output list the photos per license.

In [ ]:
%cd {WORK}
S = "alcedo-ai-mask-lab/scripts/poc/people_sky_head"
SPLITS = (("train", TRAIN_LICENSES, TRAIN_MAX_IMAGES),) if TRAIN else ()
SPLITS += (("val", VAL_LICENSES, VAL_MAX_IMAGES),)
for split, licenses, max_images in SPLITS:
    !python {S}/prepare_coco_targets.py         --panoptic-json coco/annotations/panoptic_{split}2017.json         --panoptic-png-dir coco/panoptic_{split}2017         --image-info-json coco/annotations/captions_{split}2017.json         --image-dir coco/{split}2017 --licenses {licenses} --max-images {max_images}         --out-dir splits/{split} --workers 2

## image_embed cache

SAM 2.1 tiny encoder (float32, TF32 off), float16 cache: 2 MiB per photo. The next cell checks
that the local disk has space for both caches before the encoder starts.

In [ ]:
import json, shutil
need = 0
for split, _, _ in SPLITS:
    count = len(json.load(open(f"{WORK}/splits/{split}/manifest.json"))["images"])
    need += count * 256 * 64 * 64 * 2
free = shutil.disk_usage(WORK).free
print(f"cache {need / 2**30:.1f} GiB, free {free / 2**30:.1f} GiB")
if need > free - 2 * 2**30:
    raise RuntimeError("not enough local disk for the image_embed cache: lower TRAIN_MAX_IMAGES "
                       "or VAL_MAX_IMAGES, or use a runtime with a larger disk")

In [ ]:
%cd {WORK}
for split in ("val", "train") if TRAIN else ("val",):
    !python {S}/encode_image_embed.py --split-dir splits/{split} --image-dir coco/{split}2017         --sam2-repo sam2 --checkpoint sam2.1_hiera_tiny.pt --batch-size 8 --workers 2

## Training

Checkpoints and `metrics.jsonl` go to Google Drive. Set `RESUME = True` in the settings cell to
continue from `last.pt` after a reset.

In [ ]:
%cd {WORK}
resume = "--resume" if RESUME else ""
if TRAIN:
    !python {S}/train_head.py --train-dir splits/train --val-dir splits/val --out-dir {DRIVE_OUT}/run     --epochs {EPOCHS} --batch-size {BATCH_SIZE} --workers 2 {resume}

In [ ]:
import json
for line in open(f"{DRIVE_OUT}/run/metrics.jsonl"):
    m = json.loads(line); v = m["val"]
    print(f"epoch {m['epoch']:2d} loss {m['train_loss']:.4f} | sky IoU {v['sky_cell_iou']:.3f} "
          f"| person AP50 {v['person_ap50']:.3f} top-1 {v['person_top1_hit']:.3f} "
          f"| presence AUC sky {v['presence_auc_sky']:.3f} person {v['presence_auc_person']:.3f}")

## Export

`people_sky_head.onnx` and `people_sky_head.json` (hash, validation measurements, the ONNX
Runtime against PyTorch comparison). The split manifests (photo ids and licenses) are copied to
Drive as the record of the training data.

In [ ]:
%cd {WORK}
if TRAIN:
    !python {S}/export_head_onnx.py --checkpoint {DRIVE_OUT}/run/best.pt --val-dir splits/val --out-dir {DRIVE_OUT}/export
    for split, _, _ in SPLITS:
        !mkdir -p {DRIVE_OUT}/splits/{split} && cp splits/{split}/manifest.json splits/{split}/image_embed_progress.json {DRIVE_OUT}/splits/{split}/
!ls -la {DRIVE_OUT}/export {DRIVE_OUT}/run

## Measurements of `best.pt`

`evaluate_head.py` measures `run/best.pt` on the validation split with the current
`head_metrics.py`: the People list threshold after the suppression of overlapping boxes
(`person_list_best_f1_nms`) and the box fit per person height (`person_box_fit_nms`). The result
goes to `export/people_sky_head_val_measurements.json`.

In [ ]:
%cd {WORK}
!python {S}/evaluate_head.py --checkpoint {DRIVE_OUT}/run/best.pt --val-dir splits/val --out {DRIVE_OUT}/export/people_sky_head_val_measurements.json